# core

> `Workspace`: run a kernmini shell inside IPython as a `%name`/`%%name` magic and a Python object

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
"""Shared machinery for IPython magics that embed a kernmini shell.

A kernmini shell (`execute()`/`complete()`/`inspect()`, `set_stream_sender()`, optional
`interrupt()`) normally runs inside the kernmini Rust runtime as its own Jupyter kernel.
`Workspace` runs the very same shell object inside an ordinary IPython kernel instead and
exposes it as a `%name` / `%%name` magic plus a plain Python object (`name("code")`,
`name.register(fn)`, `name.timeout`, `name.interrupt()`, `name.reset()`).

Design points, mirroring basedpl's own magics (`basedpl.notebooks`):

- A failed cell raises a typed exception (`MagicError` subclass) after any output produced
  before the error was shown; `_render_traceback_` makes IPython show the kernel's own
  message instead of Python frames.
- A trailing marker (`;` by default) suppresses display of the cell's implicit result. Output
  the code prints itself is still shown. `strip_marker` removes the marker before running,
  for languages where a trailing `;` is meaningless (Prolog, Scheme) and must stay for the
  ones where it is syntax (C, JS, Python).
- All shell calls run on one private background thread with its own event loop, so a sync
  magic can drive async shells without `asyncio.run` inside ipykernel's running loop, and a
  thread-bound engine always sees the same thread. Output events cross to the main thread
  through a queue and are printed there, so they are attributed to the running cell.
- Ctrl-C calls the shell's `interrupt()` (or cancels the task when it has none), shows the
  output captured so far, then re-raises `KeyboardInterrupt`. `timeout` (seconds, default
  off; env `<NAME>_MAGIC_TIMEOUT` or `KERNMINI_MAGIC_TIMEOUT`) does the same on a deadline and
  raises `TimeoutError`.
"""

import asyncio, atexit, inspect, os, queue, re, signal, sys, threading, time
from contextlib import contextmanager

__all__ = ['MagicError', 'Workspace']

GRACE = 3.0  # seconds an interrupted cell gets to finish and flush its output


class MagicError(RuntimeError):
    "A failed cell: keeps the shell's `{ename, evalue, traceback}` and the output shown before the error."
    def __init__(self, error=None, output=''):
        error = error or {}
        self.ename = error.get('ename') or type(self).__name__
        self.evalue = error.get('evalue') or ''
        self.kernel_traceback = list(error.get('traceback') or [])
        self.output = output
        super().__init__(self.evalue or self.ename)

    def _render_traceback_(self):
        return self.kernel_traceback or [f"{self.ename}: {self.evalue}"]


def _env_timeout(name):
    for key in (f"{name.upper()}_MAGIC_TIMEOUT", "KERNMINI_MAGIC_TIMEOUT"):
        val = os.environ.get(key, "").strip()
        if not val: continue
        try: seconds = float(val)
        except ValueError: raise ValueError(f"{key} must be a number of seconds, got {val!r}") from None
        return seconds or None
    return None


def _ipython():
    try: from IPython import get_ipython
    except ImportError: return None
    return get_ipython()


@contextmanager
def _sigint_raises():
    """Make Ctrl-C raise KeyboardInterrupt in this blocking wait.

    For magic cells ipykernel installs a handler that cancels the cell's asyncio future
    instead of raising; that only takes effect once control returns to its event loop,
    which a blocking magic never does, so Ctrl-C would hang the cell. Main thread only."""
    if threading.current_thread() is not threading.main_thread():
        yield; return
    prev = signal.getsignal(signal.SIGINT)
    signal.signal(signal.SIGINT, signal.default_int_handler)
    try: yield
    finally: signal.signal(signal.SIGINT, prev)


def _display(data, metadata=None):
    from IPython.display import display
    display(data, raw=True, metadata=metadata or None)


In [ ]:
#| export
class _Runner:
    "One daemon thread running a private asyncio loop; every shell call is scheduled on it."
    def __init__(self):
        self.loop = asyncio.new_event_loop()
        self.thread = threading.Thread(target=self._run, name='kernmini-magics', daemon=True)
        self.thread.start()

    def _run(self):
        asyncio.set_event_loop(self.loop)
        self.loop.run_forever()

    def submit(self, coro): return asyncio.run_coroutine_threadsafe(coro, self.loop)

    def stop(self):
        self.loop.call_soon_threadsafe(self.loop.stop)


async def _maybe_await(value): return await value if inspect.isawaitable(value) else value


In [ ]:
#| export
class Workspace:
    """A kernmini shell embedded in IPython: the `%name`/`%%name` magic and a Python object.

    `shell_factory()` builds the shell (sync or async), lazily on first use. `register`, if
    given, is `(shell, name, fn)` -> None/awaitable and installs a Python callable in the
    engine; omit it for languages with no host-function support.

    `call_returns` decides what `ws(code)` and the line magic return: `'result'` (the
    implicit result's text, output printed as usual) or `'output'` (the printed output
    itself, as a string, not echoed) -- for languages like Prolog whose answers are
    output, not values. The cell magic always prints.

    `aliases` registers the same magic under more names, sharing this one engine and state
    (`%%js` and `%%ts` both drive one QuickJS context)."""
    _echo = True

    def __init__(self, name, shell_factory, *, error_cls=MagicError, register=None,
                 quiet_marker=';', strip_marker=False, timeout=None, call_returns='result', aliases=()):
        if call_returns not in ('result', 'output'): raise ValueError(f"call_returns must be 'result' or 'output', not {call_returns!r}")
        self.name, self.error_cls, self.quiet_marker, self.strip_marker = name, error_cls, quiet_marker, strip_marker
        self.call_returns, self.aliases = call_returns, tuple(aliases)
        self.timeout = timeout if timeout is not None else _env_timeout(name)
        self._factory, self._register = shell_factory, register
        self._runner = self._shell = self._fut = None
        self._lock, self._atexit = threading.RLock(), False

    def __repr__(self): return f"<{self.name} workspace ({'running' if self._shell is not None else 'idle'})>"

    # -- shell lifecycle -------------------------------------------------------------
    def _ensure(self):
        with self._lock:
            if self._runner is None: self._runner = _Runner()
            if self._shell is None:
                async def make(): return await _maybe_await(self._factory())
                self._shell = self._runner.submit(make()).result()
            return self._shell

    def reset(self):
        "Drop the shell and its state; the next call builds a fresh one."
        with self._lock:
            shell, self._shell = self._shell, None
        if shell is None: return
        for attr in ('shutdown', 'close'):
            fn = getattr(shell, attr, None)
            if callable(fn):
                async def stop(fn=fn): await _maybe_await(fn())
                try: self._runner.submit(stop()).result(timeout=5)
                except Exception: pass
                break

    def close(self):
        "Reset and stop the background thread."
        self.reset()
        with self._lock:
            runner, self._runner = self._runner, None
        if runner is not None: runner.stop()

    def interrupt(self):
        "Interrupt the running cell from any thread: the shell's `interrupt()` if it has one, else cancel its task."
        shell, fut = self._shell, self._fut
        if shell is not None and callable(getattr(shell, 'interrupt', None)): shell.interrupt()
        elif fut is not None: fut.cancel()

    # -- running code ----------------------------------------------------------------
    def _prep(self, code):
        mark = self.quiet_marker
        if mark and code.rstrip().endswith(mark):
            return (code.rstrip()[:-len(mark)] if self.strip_marker else code), True
        return code, False

    def _exec_kwargs(self, shell, quiet):
        "Extra `execute()` arguments the shell declares: IPython's count, and `quiet` for shells that print results as output."
        params = inspect.signature(shell.execute).parameters
        takes_any = any(p.kind is p.VAR_KEYWORD for p in params.values())
        kw, count = {}, getattr(_ipython(), 'execution_count', None)
        if count is not None and ('execution_count' in params or takes_any): kw['execution_count'] = count
        if quiet and 'quiet' in params: kw['quiet'] = True
        return kw

    def _emit(self, item, out, echo=True):
        if item[0] == 'stream':
            _, stream, text = item
            out.append(text)
            if echo or stream == 'stderr': (sys.stderr if stream == 'stderr' else sys.stdout).write(text)
        elif item[0] == 'display':
            event = item[1]
            _display(event.get('data', event), event.get('metadata'))

    def _flush(self, q, out):
        while True:
            try: item = q.get_nowait()
            except queue.Empty: return
            if item[0] != 'done': self._emit(item, out, self._echo)

    def _drain(self, q, out):
        end = time.monotonic() + GRACE
        while time.monotonic() < end:
            try: item = q.get(timeout=0.05)
            except queue.Empty: continue
            if item[0] == 'done': return
            self._emit(item, out, self._echo)
        if self._fut is not None: self._fut.cancel()

    def _execute(self, code, show_result, quiet=False, echo=True):
        """Run `code`; print its output as it arrives (unless `echo` is False); return
        `(result mime bundle, output text)`. The result is displayed too if `show_result`."""
        shell = self._ensure()
        self._echo = echo
        q, out = queue.Queue(), []
        if hasattr(shell, 'set_stream_sender'): shell.set_stream_sender(lambda stream, text: q.put(('stream', stream, text)))
        if hasattr(shell, 'set_display_sender'): shell.set_display_sender(lambda event: q.put(('display', event)))
        fut = self._fut = self._runner.submit(shell.execute(code, **self._exec_kwargs(shell, quiet)))
        fut.add_done_callback(lambda _: q.put(('done',)))
        deadline = time.monotonic() + self.timeout if self.timeout else None
        timed_out = False
        try:
            with _sigint_raises():
                while True:
                    wait = 0.1 if deadline is None else max(0.0, min(0.1, deadline - time.monotonic()))
                    try: item = q.get(timeout=wait)
                    except queue.Empty:
                        if deadline is not None and time.monotonic() >= deadline:
                            if timed_out: fut.cancel(); break
                            timed_out, deadline = True, time.monotonic() + GRACE
                            self.interrupt()
                        continue
                    if item[0] == 'done': break
                    self._emit(item, out, echo)
        except KeyboardInterrupt:
            self.interrupt()
            self._drain(q, out)
            raise
        finally:
            self._fut = None
        self._flush(q, out)
        if timed_out: raise TimeoutError(f"{self.name} cell exceeded {self.timeout}s")
        try: res = fut.result()
        except Exception as e:
            raise self.error_cls({'ename': type(e).__name__, 'evalue': str(e)}, ''.join(out)) from e
        if res.get('error'): raise self.error_cls(res['error'], ''.join(out))
        result = res.get('result')
        if result and show_result: _display(result, res.get('result_metadata'))
        return result, ''.join(out)

    def __call__(self, code, quiet=None):
        """Run `code`. Returns the implicit result's text, with output printed (`call_returns='result'`),
        or the output text itself, not printed (`call_returns='output'`). A trailing marker or
        `quiet=True` hides the result: None, or just what the code printed itself."""
        code, marked = self._prep(code)
        quiet = bool(quiet or (quiet is None and marked))
        as_output = self.call_returns == 'output'
        result, output = self._execute(code, show_result=False, quiet=quiet, echo=not as_output)
        if as_output: return output.rstrip('\n') or None
        return None if quiet or not result else result.get('text/plain')

    def _magic(self, line, cell=None):
        if line.strip() == '--reset':
            self.reset()
            if cell is None: return
        elif cell is None: return self(line)
        code, quiet = self._prep(cell)
        self._execute(code, show_result=not quiet, quiet=quiet)

    # -- host functions, names, help ---------------------------------------------------
    def register(self, fn=None, *, name=None):
        "Make a Python callable available inside the engine; also usable as a decorator."
        if fn is None: return lambda f: self.register(f, name=name)
        if self._register is None: raise NotImplementedError(f"{self.name} has no host-function support")
        shell, nm = self._ensure(), name or fn.__name__
        async def go(): await _maybe_await(self._register(shell, nm, fn))
        self._runner.submit(go()).result()
        return fn

    def _service(self, method, *args):
        shell = self._shell
        if shell is None or not hasattr(shell, method): return None
        async def go(): return await _maybe_await(getattr(shell, method)(*args))
        try: return self._runner.submit(go()).result(timeout=2)
        except Exception: return None

    def names(self, prefix=''):
        "Completion candidates for `prefix` (empty until the first cell has run)."
        res = self._service('complete', prefix, len(prefix))
        return list(res['matches']) if res else []

    def inspect(self, name):
        "The shell's help text for `name`, or None."
        res = self._service('inspect', name, len(name))
        return (res.get('data') or {}).get('text/plain') if res and res.get('found') else None

    # -- IPython integration -----------------------------------------------------------
    def _matcher(self, name=None):
        from IPython.core.completer import context_matcher, SimpleCompletion
        name = name or self.name
        cell_re = re.compile(rf'%%{re.escape(name)}[^\S\n]*\n')
        line_re = re.compile(rf'\s*(?:\w+\s*=\s*)?%{re.escape(name)}\s+')

        @context_matcher(identifier=f'kernmini_magics.{name}')
        def complete(context):
            empty = dict(completions=[])
            # cursor_position is the column within cursor_line, not an offset into full_text
            lines = context.full_text.split('\n')
            before = '\n'.join(lines[:context.cursor_line] + [lines[context.cursor_line][:context.cursor_position]])
            if hdr := cell_re.match(before):
                code, cpos = context.full_text[hdr.end():], len(before) - hdr.end()
            elif hdr := line_re.match(context.text_until_cursor):
                code, cpos = context.text_until_cursor[hdr.end():], len(context.text_until_cursor) - hdr.end()
            else: return empty
            res = self._service('complete', code, cpos)
            if not res or not res.get('matches'): return empty
            frag = code[res.get('cursor_start', cpos):cpos]
            return dict(completions=[SimpleCompletion(m, type=f'{name} name') for m in res['matches']],
                        matched_fragment=frag, suppress=True)
        return complete

    def load(self, ip=None):
        "Register `%name`/`%%name` (and any aliases) and their completion matchers with `ip` (default: the running IPython)."
        ip = ip or _ipython()
        if ip is None: raise RuntimeError('not running inside IPython')
        matchers = ip.Completer.custom_matchers
        for name in (self.name, *self.aliases):
            ip.register_magic_function(self._magic, 'line_cell', name)
            ident = f'kernmini_magics.{name}'
            matchers[:] = [m for m in matchers if getattr(m, 'matcher_identifier', None) != ident]
            matchers.append(self._matcher(name))
        if not self._atexit: atexit.register(self.close); self._atexit = True
        return self


## Tests

A fake shell stands in for a kernmini shell, so these run without any engine.

In [ ]:
import asyncio, contextlib, io, re, time

class FakeShell:
    "Stands in for a kernmini shell: scripted results, a stream sender, an interrupt hook."
    def __init__(self):
        self.calls, self._send, self.interrupted, self.registered = [], None, False, {}
    def set_stream_sender(self, sender): self._send = sender
    def interrupt(self): self.interrupted = True
    async def execute(self, code, silent=False, store_history=True, user_expressions=None, allow_stdin=False, execution_count=None, quiet=False):
        self.calls.append(code)
        self.quiet_seen = quiet
        if code == 'boom':
            self._send('stdout', 'before error\n')
            return dict(result=None, error=dict(ename='FakeError', evalue='bad thing', traceback=['FakeError: bad thing']), user_expressions={})
        if code == 'slow':
            while not self.interrupted: await asyncio.sleep(0.01)
            self.interrupted = False
            return dict(result=None, error=None, user_expressions={})
        self._send('stdout', f'ran {code}\n')
        return dict(result={'text/plain': code.upper()}, error=None, user_expressions={})
    def complete(self, code, cursor_pos=None):
        "Like the real shells: complete the word before the cursor."
        pos = len(code) if cursor_pos is None else cursor_pos
        word = re.search(r'\w*$', code[:pos]).group()
        return dict(matches=[m for m in ('append', 'apply', 'arg') if word and m.startswith(word)],
                    cursor_start=pos - len(word), cursor_end=pos, status='ok')

class FakeError(MagicError): pass

def reg(shell, name, fn): shell.registered[name] = fn

def new(**kw): return Workspace('fake', FakeShell, error_cls=FakeError, register=reg, **kw)

def capture(fn, *a, **kw):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf): res = fn(*a, **kw)
    return res, buf.getvalue()

# output is printed, the implicit result's text is returned, and a trailing `;` hides only the result
ws = new()
res, printed = capture(ws, 'hello')
assert (res, printed) == ('HELLO', 'ran hello\n'), (res, printed)
res, printed = capture(ws, 'hello;')
assert (res, printed) == (None, 'ran hello;\n'), (res, printed)

# a quiet cell tells the shell, which can then hide results it prints as output
ws.quiet_seen = None
capture(ws, 'hello;')
assert ws._shell.quiet_seen is True
capture(ws, 'hello')
assert ws._shell.quiet_seen is False

# call_returns='output' returns what was printed, without echoing it; the cell magic still prints
wo = new(call_returns='output')
assert capture(wo, 'hello') == ('ran hello', '')
buf = io.StringIO()
with contextlib.redirect_stdout(buf): wo._magic('', 'hello')
assert buf.getvalue() == 'ran hello\n'
try: new(call_returns='nope')
except ValueError: pass
else: raise AssertionError('bad call_returns accepted')
wo.close()

# strip_marker removes the marker before the code runs
ws2 = new(strip_marker=True)
capture(ws2, 'hello;')
assert ws2._shell.calls == ['hello']

# an error raises the typed exception, after the output printed before it
try: capture(ws, 'boom')
except FakeError as e:
    assert (e.ename, e.evalue, e.output) == ('FakeError', 'bad thing', 'before error\n')
    assert e._render_traceback_() == ['FakeError: bad thing']
else: raise AssertionError('no error raised')

# the shell keeps its state after an error
assert capture(ws, 'again')[0] == 'AGAIN'

# register works as a decorator and as a call, and the hook sees the live shell
@ws.register
def double(x): return 2 * x
ws.register(len, name='length')
assert set(ws._shell.registered) == {'double', 'length'}
try: Workspace('nohost', FakeShell).register(len)
except NotImplementedError: pass
else: raise AssertionError('register should need a hook')

# a timeout interrupts through the shell's hook and raises TimeoutError; the workspace stays usable
ws3 = new(timeout=0.3)
t0 = time.monotonic()
try: capture(ws3, 'slow')
except TimeoutError: pass
else: raise AssertionError('no timeout')
assert time.monotonic() - t0 < 2
assert capture(ws3, 'ok')[0] == 'OK'

# names() comes from the shell's complete(); reset() builds a fresh shell
assert ws.names('ap') == ['append', 'apply']
first = ws._shell
ws.reset()
assert ws._shell is None and capture(ws, 'x')[0] == 'X' and ws._shell is not first

# the timeout comes from the environment unless set explicitly
import os
os.environ['KERNMINI_MAGIC_TIMEOUT'] = '7'
assert new().timeout == 7.0 and new(timeout=1).timeout == 1
os.environ['FAKE_MAGIC_TIMEOUT'] = '0'
assert new().timeout is None
del os.environ['KERNMINI_MAGIC_TIMEOUT'], os.environ['FAKE_MAGIC_TIMEOUT']
# the completion matcher reads the code after the magic header, using line + column cursor coordinates
from IPython.core.completer import CompletionContext
def complete(text):
    lines = text.split('\n')
    ctx = CompletionContext(token='', full_text=text, cursor_position=len(lines[-1]), cursor_line=len(lines)-1, limit=None)
    res = ws._matcher()(ctx)
    return [c.text for c in res['completions']], res.get('matched_fragment')
ws('warm')
assert complete('%%fake\nfoo\nap') == (['append', 'apply'], 'ap')
assert complete('x = %fake ap') == (['append', 'apply'], 'ap')
assert complete('%%other\nap') == ([], None)

# an alias completes through the same shell
wa = new(aliases=('fake2',))
wa('warm')
lines = '%%fake2\nap'.split('\n')
ctx = CompletionContext(token='', full_text='%%fake2\nap', cursor_position=2, cursor_line=1, limit=None)
assert [c.text for c in wa._matcher('fake2')(ctx)['completions']] == ['append', 'apply']
wa.close()
assert complete('ap') == ([], None)

for w in (ws, ws2, ws3): w.close()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()